# WAV to Mel Spectrogram `.npy` Converter

This notebook has one responsibility only:
1. Read split manifests (`train/val/test` CSV files).
2. Resolve and load audio files listed in those CSVs.
3. Convert each file directly to mel spectrogram (no waveform or mel preprocessing).
4. Save the mel spectrogram as `.npy` in output split folders.

No labeling logic, manifest generation, or publishing steps are included.

## 1) Imports

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
from tqdm.auto import tqdm

print("Imports ready.")

## 2) Configuration

In [ ]:
@dataclass
class CFG:
    # Root folder of audio data used to resolve relative CSV paths.
    input_root: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/data"

    # Split CSVs (source of train/val/test assignment).
    train_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/train_split.csv"
    val_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/val_split.csv"
    test_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/test_split.csv"

    # Output root folder where split .npy mel files will be written.
    output_root: str = "/kaggle/working/mel_npy_output"

    # Mel spectrogram parameters (high-detail defaults).
    n_fft: int = 4096
    hop_length: int = 256
    win_length: int = 4096
    n_mels: int = 256
    f_min: float = 0.0
    f_max: float | None = None
    power: float = 2.0
    normalized: bool = False
    center: bool = True
    pad_mode: str = "reflect"

    # Output dtype for saved arrays
    output_dtype: str = "float32"  # float32 or float16

    # Use GPU for mel conversion when CUDA is available
    use_gpu: bool = True

    # Recompute files even if destination exists
    overwrite: bool = False

cfg = CFG()

input_root = Path(cfg.input_root)
output_root = Path(cfg.output_root)
csv_paths = {
    "train": Path(cfg.train_csv),
    "val": Path(cfg.val_csv),
    "test": Path(cfg.test_csv),
}

if not input_root.exists():
    raise FileNotFoundError(f"Input root not found: {input_root}")
for split_name, csv_path in csv_paths.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {split_name} CSV: {csv_path}")

output_root.mkdir(parents=True, exist_ok=True)

if cfg.output_dtype not in {"float32", "float16"}:
    raise ValueError("output_dtype must be one of: float32, float16")

print(json.dumps(asdict(cfg), indent=2))

## 3) Split and convert audio to mel `.npy`

In [ ]:
path_candidates = ["file_path", "filepath", "path", "audio_path"]

def get_path_column(df: pd.DataFrame) -> str:
    for col in path_candidates:
        if col in df.columns:
            return col
    raise ValueError(f"CSV must contain one of {path_candidates}. Found: {list(df.columns)}")

# Build basename index once for robust path recovery (same style as Kaggle notebook).
all_wavs = list(input_root.rglob("*.wav"))
basename_to_paths = {}
for p in all_wavs:
    basename_to_paths.setdefault(p.name, []).append(p)

def resolve_audio_path(raw_path: str) -> Path:
    s = str(raw_path).replace('\\', '/').strip()

    p = Path(s)
    if p.exists():
        return p

    parts = s.split('/')
    if 'data' in parts:
        idx = parts.index('data')
        candidate = input_root / Path(*parts[idx + 1:])
        if candidate.exists():
            return candidate

    for machine_name in ['Machine 1', 'Machine 2', 'Machine 3']:
        marker = f'/{machine_name}/'
        if marker in s:
            suffix = s.split(marker, 1)[1]
            candidate = input_root / machine_name / suffix
            if candidate.exists():
                return candidate

    if not p.is_absolute():
        candidate = input_root / p
        if candidate.exists():
            return candidate

    base = Path(s).name
    matches = basename_to_paths.get(base, [])
    if len(matches) == 1:
        return matches[0]

    return Path(s)

rows = []
for split_name, csv_path in csv_paths.items():
    df = pd.read_csv(csv_path)
    path_col = get_path_column(df)
    values = [str(v).strip() for v in df[path_col].dropna().tolist()]
    values = [v for v in values if v]
    for raw in values:
        rows.append({"split": split_name, "raw_path": raw})

if not rows:
    raise ValueError("No rows found across split CSV files")

print(f"Rows from CSVs: {len(rows)}")

device = torch.device("cuda" if (cfg.use_gpu and torch.cuda.is_available()) else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Compute device: {device}")

dtype_map = {"float32": np.float32, "float16": np.float16}
np_dtype = dtype_map[cfg.output_dtype]

mel_transform_cache: dict[int, T.MelSpectrogram] = {}

def get_mel_transform(sample_rate: int) -> T.MelSpectrogram:
    if sample_rate not in mel_transform_cache:
        f_max = cfg.f_max if cfg.f_max is not None else sample_rate / 2
        mel_transform_cache[sample_rate] = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=cfg.n_fft,
            win_length=cfg.win_length,
            hop_length=cfg.hop_length,
            f_min=cfg.f_min,
            f_max=f_max,
            n_mels=cfg.n_mels,
            power=cfg.power,
            normalized=cfg.normalized,
            center=cfg.center,
            pad_mode=cfg.pad_mode,
        ).to(device)
    return mel_transform_cache[sample_rate]

processed = 0
skipped = 0
errors = []
processed_by_split = {"train": 0, "val": 0, "test": 0}

for row in tqdm(rows, desc="Converting"):
    split_name = row["split"]
    src_path = resolve_audio_path(row["raw_path"])

    if not src_path.exists():
        errors.append((row["raw_path"], "Resolved path does not exist"))
        continue

    try:
        rel = src_path.relative_to(input_root)
        dst_path = output_root / "splits" / split_name / rel.with_suffix(".npy")
    except ValueError:
        dst_path = output_root / "splits" / split_name / f"{src_path.stem}.npy"

    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if dst_path.exists() and not cfg.overwrite:
        skipped += 1
        continue

    try:
        waveform, sample_rate = torchaudio.load(str(src_path))
        waveform = waveform.to(device, non_blocking=True)
        mel_transform = get_mel_transform(sample_rate)
        with torch.inference_mode():
            mel = mel_transform(waveform)
        mel_np = mel.detach().cpu().numpy().astype(np_dtype, copy=False)
        np.save(dst_path, mel_np)
        processed += 1
        processed_by_split[split_name] += 1
    except Exception as e:
        errors.append((str(src_path), str(e)))

print(f"Processed: {processed}")
print(f"Skipped:   {skipped}")
print(f"Errors:    {len(errors)}")
print("Processed by split:", processed_by_split)
if errors:
    print("Sample errors:")
    for p, msg in errors[:10]:
        print(f"- {p}: {msg}")

print(f"Output root: {output_root / 'splits'}")

## 4) Visual check: original mel vs saved-and-reloaded `.npy` mel

Run this after conversion to verify the stored `.npy` reproduces the same mel spectrogram values.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torchaudio
import torchaudio.transforms as T

# Set your target audio sample path here.
sample_audio_path = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/data/Machine 1/machine_data/Normal/10018.wav"

# Standalone mel config (matches your conversion defaults).
mel_cfg = {
    "n_fft": 4096,
    "hop_length": 256,
    "win_length": 4096,
    "n_mels": 256,
    "f_min": 0.0,
    "f_max": None,
    "power": 2.0,
    "normalized": False,
    "center": True,
    "pad_mode": "reflect",
    "output_dtype": "float32",  # float32 or float16
}

dtype_map = {"float32": np.float32, "float16": np.float16}
if mel_cfg["output_dtype"] not in dtype_map:
    raise ValueError("output_dtype must be one of: float32, float16")
np_dtype = dtype_map[mel_cfg["output_dtype"]]

def build_mel_transform(sample_rate: int) -> T.MelSpectrogram:
    f_max = mel_cfg["f_max"] if mel_cfg["f_max"] is not None else sample_rate / 2
    return T.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=mel_cfg["n_fft"],
        win_length=mel_cfg["win_length"],
        hop_length=mel_cfg["hop_length"],
        f_min=mel_cfg["f_min"],
        f_max=f_max,
        n_mels=mel_cfg["n_mels"],
        power=mel_cfg["power"],
        normalized=mel_cfg["normalized"],
        center=mel_cfg["center"],
        pad_mode=mel_cfg["pad_mode"],
    )

src_path = Path(sample_audio_path)
if not src_path.exists():
    raise FileNotFoundError(f"Audio file not found: {src_path}")

# Create mel spectrogram.
waveform, sample_rate = torchaudio.load(str(src_path))
mel_transform = build_mel_transform(sample_rate)
mel_original = mel_transform(waveform)[0].cpu().numpy().astype(np_dtype, copy=False)

# Save mel as .npy, then load it back.
npy_path = Path("/kaggle/working") / f"{src_path.stem}_mel.npy"
np.save(npy_path, mel_original)
mel_from_npy = np.load(npy_path)

# Display both as dB for easier visual comparison.
eps = 1e-10
mel_original_db = 10.0 * np.log10(np.maximum(mel_original, eps))
mel_from_npy_db = 10.0 * np.log10(np.maximum(mel_from_npy, eps))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(mel_original_db, aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("Original Mel (dB)")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Mel bins")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(mel_from_npy_db, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("Mel Loaded From NPY (dB)")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Mel bins")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Audio path:   {src_path}")
print(f"NPY path:     {npy_path}")
print(f"Original shape: {mel_original.shape}, Loaded shape: {mel_from_npy.shape}")
print("Max abs diff (linear):", float(np.max(np.abs(mel_original - mel_from_npy))))

## 5) Zip output and delete original folder

Run this last, after conversion is complete and you have verified outputs.

In [ ]:
import shutil
from pathlib import Path

src_dir = Path('/kaggle/working/mel_npy_output')
zip_base = '/kaggle/working/mel_output'

if not src_dir.exists():
    raise FileNotFoundError(f'Folder not found: {src_dir}')

zip_path = shutil.make_archive(zip_base, 'zip', str(src_dir))
print(f'Created zip: {zip_path}')

shutil.rmtree(src_dir)
print(f'Removed folder: {src_dir}')